In [0]:
dbutils.library.restartPython() 


In [0]:
%pip install datasets huggingface_hub

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import datasets
import huggingface_hub
print("✅ datasets version:", datasets.__version__)
print("✅ huggingface_hub version:", huggingface_hub.__version__)

✅ datasets version: 4.6.1
✅ huggingface_hub version: 1.5.0


In [0]:
from huggingface_hub import hf_hub_download
import pandas as pd

try:
    # Télécharger le fichier parquet directement
    file_path = hf_hub_download(
        repo_id="McAuley-Lab/Amazon-Reviews-2023",
        filename="raw_review_All_Beauty/full.parquet",
        repo_type="dataset"
    )
    df_pandas = pd.read_parquet(file_path)
    print(f"✅ Dataset chargé : {len(df_pandas):,} interactions")
    print(f"Colonnes : {df_pandas.columns.tolist()}")
    display(df_pandas.head(3))
except Exception as e:
    print("❌ Échec du téléchargement ou du chargement du fichier :", e)


❌ Échec du téléchargement ou du chargement du fichier : 404 Client Error. (Request ID: Root=1-69a8196f-0adc69b744ef245611583f61;0432d3b2-c5a7-4868-ab16-596bffb3e56a)

Entry Not Found for url: https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw_review_All_Beauty/full.parquet.


Lister tous les fichiers disponibles dans ce dataset

In [0]:
from huggingface_hub import HfApi

api = HfApi()

# Lister tous les fichiers disponibles dans ce dataset
files = api.list_repo_files(
    repo_id="McAuley-Lab/Amazon-Reviews-2023",
    repo_type="dataset"
)

# Afficher tous les fichiers disponibles
for f in files:
    print(f)

.gitattributes
Amazon-Reviews-2023.py
README.md
all_categories.txt
asin2category.json
benchmark/0core/last_out/All_Beauty.test.csv
benchmark/0core/last_out/All_Beauty.train.csv
benchmark/0core/last_out/All_Beauty.valid.csv
benchmark/0core/last_out/Amazon_Fashion.test.csv
benchmark/0core/last_out/Amazon_Fashion.train.csv
benchmark/0core/last_out/Amazon_Fashion.valid.csv
benchmark/0core/last_out/Appliances.test.csv
benchmark/0core/last_out/Appliances.train.csv
benchmark/0core/last_out/Appliances.valid.csv
benchmark/0core/last_out/Arts_Crafts_and_Sewing.test.csv
benchmark/0core/last_out/Arts_Crafts_and_Sewing.train.csv
benchmark/0core/last_out/Arts_Crafts_and_Sewing.valid.csv
benchmark/0core/last_out/Automotive.test.csv
benchmark/0core/last_out/Automotive.train.csv
benchmark/0core/last_out/Automotive.valid.csv
benchmark/0core/last_out/Baby_Products.test.csv
benchmark/0core/last_out/Baby_Products.train.csv
benchmark/0core/last_out/Baby_Products.valid.csv
benchmark/0core/last_out/Beauty_and

 Télécharger les reviews (JSONL)
 

In [0]:
from huggingface_hub import hf_hub_download
import pandas as pd

# Télécharger le fichier JSONL des reviews All_Beauty
reviews_path = hf_hub_download(
    repo_id="McAuley-Lab/Amazon-Reviews-2023",
    filename="raw/review_categories/All_Beauty.jsonl",
    repo_type="dataset"
)

print(f"✅ Fichier téléchargé : {reviews_path}")

raw/review_categories/All_Beauty.jsonl:   0%|          | 0.00/327M [00:00<?, ?B/s]

✅ Fichier téléchargé : /home/spark-d124110d-66c4-4283-9082-cd/.cache/huggingface/hub/datasets--McAuley-Lab--Amazon-Reviews-2023/snapshots/2b6d039ed471f2ba5fd2acb718bf33b0a7e5598e/raw/review_categories/All_Beauty.jsonl


 Lire le JSONL et convertir en Spark DataFrame
 

In [0]:
import pandas as pd  # minimal fix for NameError
# Lire le fichier JSONL ligne par ligne
df_reviews_pandas = pd.read_json(reviews_path, lines=True)

print(f"✅ Nombre de reviews : {len(df_reviews_pandas):,}")
print(f"Colonnes : {df_reviews_pandas.columns.tolist()}")
print(df_reviews_pandas.head(3))

✅ Nombre de reviews : 701,528
Colonnes : ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']
   rating  ... verified_purchase
0       5  ...              True
1       4  ...              True
2       5  ...              True

[3 rows x 10 columns]


Vérifier les catalogues disponibles


In [0]:
spark.sql("SHOW CATALOGS").show()

+---------+
|  catalog|
+---------+
|  samples|
|   system|
|workspace|
+---------+



Vérifier les schémas du catalogue workspace

In [0]:
spark.sql("SHOW SCHEMAS IN workspace").show()


+------------------+
|      databaseName|
+------------------+
|           default|
|information_schema|
+------------------+



Créer un volume pour datasets

In [0]:
spark.sql("""
CREATE VOLUME workspace.default.ecommerce_data
""")

DataFrame[]

Sauvegarder les reviews

In [0]:
df_reviews_spark = spark.createDataFrame(df_reviews_pandas)

df_reviews_spark.write \
    .format("parquet") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/ecommerce_data/reviews_beauty")

print("✅ Reviews sauvegardées !")

display(df_reviews_spark.limit(5))

✅ Reviews sauvegardées !


rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
5,Such a lovely scent but not overpowering.,"This spray is really nice. It smells really good, goes on really fine, and does the trick. I will say it feels like you need a lot of it though to get the texture I want. I have a lot of hair, medium thickness. I am comparing to other brands with yucky chemicals so I'm gonna stick with this. Try it!",List(),B00YQ6X8EO,B00YQ6X8EO,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2020-05-05T14:08:48.923Z,0,true
4,Works great but smells a little weird.,"This product does what I need it to do, I just wish it was odorless or had a soft coconut smell. Having my head smell like an orange coffee is offputting. (granted, I did know the smell was described but I was hoping it would be light)",List(),B081TJ8YS3,B081TJ8YS3,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2020-05-04T18:10:55.070Z,1,true
5,Yes!,"Smells good, feels great!",List(),B07PNNCSP9,B097R46CSY,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,2020-05-16T21:41:06.052Z,2,true
1,Synthetic feeling,Felt synthetic,List(),B09JS339BZ,B09JS339BZ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,2022-01-28T18:13:50.220Z,0,true
5,A+,Love it,List(),B08BZ63GMJ,B08BZ63GMJ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,2020-12-30T10:02:43.534Z,0,true


Télécharger les métadonnées

In [0]:
# Télécharger le fichier parquet des métadonnées
meta_path = hf_hub_download(
    repo_id="McAuley-Lab/Amazon-Reviews-2023",
    filename="raw_meta_All_Beauty/full-00000-of-00001.parquet",
    repo_type="dataset"
)
print(f"✅ Métadonnées téléchargées : {meta_path}")

# Corriger : copier le fichier téléchargé vers un volume accessible (Uniquement si nécessaire)
import shutil
import os

destination = "/Volumes/workspace/default/ecommerce_data/full-00000-of-00001.parquet"
shutil.copy(meta_path, destination)

# Lire directement avec Spark depuis le volume
df_meta_spark = spark.read.parquet(destination)

# Sauvegarder dans le même Volume que les reviews
df_meta_spark.write \
    .format("parquet") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/ecommerce_data/metadata_beauty")

print(f"✅ Métadonnées sauvegardées : {df_meta_spark.count():,} produits")
display(df_meta_spark.limit(5))

✅ Métadonnées téléchargées : /home/spark-d124110d-66c4-4283-9082-cd/.cache/huggingface/hub/datasets--McAuley-Lab--Amazon-Reviews-2023/snapshots/2b6d039ed471f2ba5fd2acb718bf33b0a7e5598e/raw_meta_All_Beauty/full-00000-of-00001.parquet
✅ Métadonnées sauvegardées : 112,590 produits


main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
All Beauty,"Howard LC0008 Leather Conditioner, 8-Ounce (4-Pack)",4.8,10,List(),List(),None,"List(List(null, https://m.media-amazon.com/images/I/71i77AuI9xL._SL1500_.jpg), List(https://m.media-amazon.com/images/I/41qfjSfqNyL.jpg, https://m.media-amazon.com/images/I/41w2yznfuZL.jpg), List(https://m.media-amazon.com/images/I/41qfjSfqNyL._SS40_.jpg, https://m.media-amazon.com/images/I/41w2yznfuZL._SS40_.jpg), List(MAIN, PT01))","List(List(), List(), List())",Howard Products,List(),"{""Package Dimensions"": ""7.1 x 5.5 x 3 inches; 2.38 Pounds"", ""UPC"": ""617390882781""}",B01CUPMQZE,null,null,null
All Beauty,"Yes to Tomatoes Detoxifying Charcoal Cleanser (Pack of 2) with Charcoal Powder, Tomato Fruit Extract, and Gingko Biloba Leaf Extract, 5 fl. oz.",4.5,3,List(),List(),None,"List(List(https://m.media-amazon.com/images/I/71g1lP0pMbL._SL1500_.jpg, https://m.media-amazon.com/images/I/81OqvR94isL._SL1500_.jpg), List(https://m.media-amazon.com/images/I/41b+11d5igL.jpg, https://m.media-amazon.com/images/I/41j2ocUzCtL.jpg), List(https://m.media-amazon.com/images/I/41b+11d5igL._SS40_.jpg, https://m.media-amazon.com/images/I/41j2ocUzCtL._SS40_.jpg), List(MAIN, PT01))","List(List(), List(), List())",Yes To,List(),"{""Item Form"": ""Powder"", ""Skin Type"": ""Acne Prone"", ""Brand"": ""Yes To"", ""Age Range (Description)"": ""Adult"", ""Unit Count"": ""10 Fl Oz"", ""Is Discontinued By Manufacturer"": ""No"", ""Item model number"": ""SG_B076WQZGPM_US"", ""UPC"": ""653801351125"", ""Manufacturer"": ""Yes to Tomatoes""}",B076WQZGPM,null,null,null
All Beauty,Eye Patch Black Adult with Tie Band (6 Per Pack),4.4,26,List(),List(),None,"List(List(null, null), List(https://m.media-amazon.com/images/I/31bz+uqzWCL.jpg, https://m.media-amazon.com/images/I/31bz+uqzWCL.jpg), List(https://m.media-amazon.com/images/I/31bz+uqzWCL._SS40_.jpg, https://m.media-amazon.com/images/I/31bz+uqzWCL._SS40_.jpg), List(MAIN, PT01))","List(List(), List(), List())",Levine Health Products,List(),"{""Manufacturer"": ""Levine Health Products""}",B000B658RI,null,null,null
All Beauty,"Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4D Imitation Eyebrow Tattoos, 4D Hair-like Authentic Eyebrows Waterproof Long Lasting for Woman & Man Makeup Tool",3.1,102,List(),List(),None,"List(List(https://m.media-amazon.com/images/I/71GJhXQGvyL._SL1500_.jpg, https://m.media-amazon.com/images/I/61NS1lONhzL._SL1001_.jpg, https://m.media-amazon.com/images/I/61lcwXtw3ZL._SL1001_.jpg, https://m.media-amazon.com/images/I/61G-iZeX-LL._SL1001_.jpg, https://m.media-amazon.com/images/I/618BBBsAQrL._SL1001_.jpg, https://m.media-amazon.com/images/I/61iomcZjbAL._SL1001_.jpg, https://m.media-amazon.com/images/I/61m-71vCbCL._SL1001_.jpg), List(https://m.media-amazon.com/images/I/515iwxdKS1L.jpg, https://m.media-amazon.com/images/I/51ALSVnGDML.jpg, https://m.media-amazon.com/images/I/51GvPall-ML.jpg, https://m.media-amazon.com/images/I/51JFEUql-KL.jpg, https://m.media-amazon.com/images/I/414rIT4vNAL.jpg, https://m.media-amazon.com/images/I/41JVLUdtYaL.jpg, https://m.media-amazon.com/images/I/51Vxn6nVrVL.jpg), List(https://m.media-amazon.com/images/I/515iwxdKS1L._SS40_.jpg, https://m.media-amazon.com/images/I/51ALSVnGDML._SS40_.jpg, https://m.media-amazon.com/images/I/51GvPall-ML._SS40_.jpg, https://m.media-amazon.com/images/I/51JFEUql-KL._SS40_.jpg, https://m.media-amazon.com/images/I/414rIT4vNAL._SS40_.jpg, https://m.media-amazon.com/images/I/41JVLUdtYaL._SS40_.jpg, https://m.media-amazon.com/images/I/51Vxn6nVrVL._SS40_.jpg), List(MAIN, PT01, PT02, PT03, PT04, PT05, PT06))","List(List(), List(), List())",Cherioll,List(),"{""Brand"": ""Cherioll"", ""Item Form"": ""Powder"", ""Finish Type"": ""Natural"", ""Product Benefits"": ""Long Lasting"", ""Skin Type"": ""All"", ""Package Dimensions"": ""8.43 x 5.91 x 0.87 inches; 8.78 Ounces"", ""Item model number""